# BIOT 6900 · Module 7 · Assignment Starter (Neo4j)

### *Build a knowledge graph for YOUR Part I disease*

This is the starter for the graded assignment. The database plumbing, the Cypher
queries, and the graph algorithms are **already written and working** — your job
is to supply the **data for your own disease** (the cells marked **`# TODO`**) and
then **interpret** what the graph tells you.

It ships with a tiny placeholder example so it runs top to bottom as a skeleton.
Replace the example with your disease, section by section, re-running as you go.

**Before you start:** create a free Neo4j AuraDB instance and have your
`targets_ad.csv` from Module 2 ready. See the Module 7 lab guide for setup.
Run with **Kernel → Restart & Run All**.


## 0 · Install and connect

Use the exact `NEO4J_URI`, `NEO4J_USERNAME`, and `NEO4J_PASSWORD` from your
downloaded Aura credentials file. The connection cell reads `Module_7/.env`
(git-ignored), with environment variables taking precedence. If only the password
is missing, it requests it with a hidden prompt. Credentials are never printed.
Use the account's home database unless its credentials explicitly specify another
database; do not assume it is named `neo4j`.


In [1]:
# Install only missing packages, so Restart & Run All does not repeat network setup.
import importlib.util
packages = {"neo4j": "neo4j", "networkx": "networkx", "matplotlib": "matplotlib",
            "pandas": "pandas", "requests": "requests", "scipy": "scipy",
            "reportlab": "reportlab", "pymupdf": "pymupdf"}
missing_packages = [package for module, package in packages.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    get_ipython().run_line_magic("pip", "install " + " ".join(missing_packages))
else:
    print("All required packages are already installed.")


All required packages are already installed.


In [2]:
import os
import json
import logging
from getpass import getpass
from pathlib import Path
from urllib.parse import urlsplit
from neo4j import GraphDatabase

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "module_2/targets_luad.csv").is_file())
ARTIFACT_DIR = PROJECT_ROOT / "Module_7"
_local_config = {}
_config_path = ARTIFACT_DIR / ".env"
if _config_path.is_file():
    for line in _config_path.read_text(encoding="utf-8-sig").splitlines():
        if not line.strip() or line.lstrip().startswith("#"):
            continue
        key, sep, value = line.partition("=")
        if sep:
            value = value.strip()
            _local_config[key.strip()] = (json.loads(value) if value.startswith('"')
                                          else value.strip("'"))

def config(key):
    return os.environ.get(key, _local_config.get(key, ""))

_old_driver = globals().get("driver")
driver = None
if _old_driver is not None:
    _old_driver.close()

def run_query(cypher, **params):
    if driver is None:
        raise RuntimeError("Run the connection cell successfully before running database queries.")
    options = {"database": NEO4J_DATABASE} if NEO4J_DATABASE else {}
    with driver.session(**options) as session:
        return [record.data() for record in session.run(cypher, **params)]

NEO4J_URI = config("NEO4J_URI").strip()
NEO4J_USER = (config("NEO4J_USERNAME") or config("NEO4J_USER")).strip()
NEO4J_PASSWORD = config("NEO4J_PASSWORD")
NEO4J_DATABASE = config("NEO4J_DATABASE").strip() or None
if not NEO4J_URI or not NEO4J_USER:
    raise ValueError("Configure the exact Aura URI and username in Module_7/.env or environment variables.")
if not NEO4J_PASSWORD:
    NEO4J_PASSWORD = getpass("Aura password (hidden; not saved): ")
if not NEO4J_PASSWORD:
    raise ValueError("A password is required. Use the local configuration or hidden prompt.")
_uri = urlsplit(NEO4J_URI)
if _uri.scheme not in {"neo4j", "neo4j+s", "neo4j+ssc", "bolt", "bolt+s", "bolt+ssc"} or not _uri.hostname:
    raise ValueError("Copy the complete connection URI from your Aura credentials file.")

# Keep credentials and private connection details out of exception/log output.
logging.getLogger("neo4j").setLevel(logging.CRITICAL)
print("Connecting to Neo4j...", flush=True)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD),
                              connection_timeout=10, connection_acquisition_timeout=15)
try:
    driver.verify_connectivity()
    run_query("RETURN 1 AS connection_check")
except Exception as exc:
    _failed_driver, driver = driver, None
    _failed_driver.close()
    _code = getattr(exc, "code", "") or type(exc).__name__
    raise ConnectionError("Aura connection failed (" + _code +
                          "). Check your local credentials, database selection, and network.") from None
print("Connected to Neo4j.")


Connecting to Neo4j...


Connected to Neo4j.


## 1 · Your data  (★ this is the part you edit)

Replace every value marked **`# TODO`** with your own disease. Read your genes
from `targets_ad.csv`, then fill in the phenotypes, pathways, drugs, variants, and
**at least one related disease that shares a phenotype** (that is what powers the
repurposing query in §4).

In [3]:
import pandas as pd
import numpy as np
import networkx as nx
from collections import Counter, defaultdict
from IPython.display import display

# Existing project measurements and predictions; no new cohort data are generated.
DISEASE = "Lung adenocarcinoma (LUAD)"
PROJECT = "BIOT6900_Module7_LUAD"
EXPECTED_GENES = ["ACTN2", "ZNF536", "COL6A6", "TNR", "SCN7A", "ABCA8", "ADAMTS16",
                  "TNXB", "NOS1", "MYH2", "MASP1", "ERICH3", "COL11A1", "SEMA5A", "GDF10"]
targets = pd.read_csv(PROJECT_ROOT / "module_2/targets_luad.csv")
integrated = pd.read_csv(PROJECT_ROOT / "module_2/data/luad_integrated_gene_level.csv")
week3 = pd.read_csv(PROJECT_ROOT / "module_3/targets_luad_w3.csv")
mapping = pd.read_csv(PROJECT_ROOT / "module_3/validation/target_mapping_and_scores.csv")
structure = pd.read_csv(PROJECT_ROOT / "module_4/results/verified_structure_summary.csv")
pgx = pd.read_csv(PROJECT_ROOT / "module_5/pgx_classified_CYP2C19.csv")
patient_metadata = pd.read_csv(PROJECT_ROOT / "module_6/data/meta.csv")
module6_manifest = json.loads((PROJECT_ROOT / "module_6/data/data_preparation_manifest.json").read_text())
assert targets["gene"].tolist() == EXPECTED_GENES
assert set(week3["gene"]) == set(EXPECTED_GENES)
cohort = targets.merge(integrated[["gene", "ensembl", "rna_n_pairs", "prot_n_pairs", "rna_q_bh", "prot_q_bh", "module2_rank"]],
                       on="gene", validate="one_to_one")
cohort = cohort.merge(week3[["gene", "rank", "p_sm", "p_ab", "final"]].rename(
    columns={"rank": "module3_rank", "final": "module3_final"}), on="gene", validate="one_to_one")
cohort = cohort.merge(mapping[["gene", "accession", "embedding_truncated"]], on="gene", validate="one_to_one")
assert len(cohort) == 15 and cohort["gene"].is_unique

# Authoritative identifiers and edge-specific references verified 2026-11-02.
CATALOG = {'retrieved': '2026-11-02', 'SOURCES': {'nci_nsclc': {'title': 'Non-Small Cell Lung Cancer Treatment (PDQ) - Patient Version', 'author': 'National Cancer Institute', 'year': 'n.d.', 'url': 'https://www.cancer.gov/types/lung/patient/non-small-cell-lung-treatment-pdq', 'support': 'Identifies adenocarcinoma as an NSCLC histology and describes persistent cough and difficulty breathing. Symptoms are clinical context for NSCLC, not a frequency estimate specific to this LUAD cohort.', 'retrieved': '2026-11-02'}, 'nci_sclc': {'title': 'Small Cell Lung Cancer Treatment (PDQ) - Patient Version', 'author': 'National Cancer Institute', 'year': 'n.d.', 'url': 'https://www.cancer.gov/types/lung/patient/small-cell-lung-treatment-pdq', 'support': 'Describes persistent cough and difficulty breathing as SCLC symptoms and lists topotecan among chemotherapy options.', 'retrieved': '2026-11-02'}, 'fda_osimertinib': {'title': 'Tagrisso (osimertinib) tablets: Prescribing information', 'author': 'U.S. Food and Drug Administration', 'year': 2024, 'url': 'https://www.accessdata.fda.gov/drugsatfda_docs/label/2024/208065s033lbl.pdf', 'support': 'Sections 1 and 12.1 support approved uses in specified EGFR-mutant NSCLC populations and inhibition of mutant EGFR, including L858R. This supports contextual EGFR biology and a drug-target edge, not treatment eligibility for every LUAD case. Label revision September 2024; not represented as the latest label.', 'retrieved': '2026-11-02'}, 'fda_topotecan': {'title': 'Topotecan injection: Prescribing information', 'author': 'U.S. Food and Drug Administration', 'year': 2019, 'url': 'https://www.accessdata.fda.gov/drugsatfda_docs/label/2019/200582s003lbl.pdf', 'support': 'Section 1 documents an SCLC indication in platinum-sensitive disease after progression; section 12.1 describes trapping of the topoisomerase I-DNA complex. TOP1-to-SCLC is therapeutic-target context, not a claim that TOP1 is an SCLC driver. No LUAD efficacy is established by this path. Label revision December 2019; not represented as the latest label.', 'retrieved': '2026-11-02'}, 'reactome_ecm': {'title': 'Extracellular matrix organization (R-HSA-1474244)', 'author': 'Reactome', 'year': 'n.d.', 'url': 'https://reactome.org/content/detail/R-HSA-1474244', 'support': 'Curated human pathway. Human physical-entity pages verify membership of COL6A6, COL11A1, TNXB, and TNR. General matrix remodeling biology supports a hypothesis about tissue composition or stromal biology, not a validated LUAD subtype.', 'retrieved': '2026-11-02'}, 'reactome_lectin': {'title': 'Lectin pathway of complement activation (R-HSA-166662)', 'author': 'Reactome', 'year': 'n.d.', 'url': 'https://reactome.org/content/detail/R-HSA-166662', 'support': 'Human pathway summary directly describes MASP1 involvement in protease activation and complement signaling. This is pathway membership, not evidence that MASP1 causes LUAD.', 'retrieved': '2026-11-02'}, 'reactome_egfr': {'title': 'Signaling by EGFR (R-HSA-177929)', 'author': 'Reactome', 'year': 'n.d.', 'url': 'https://reactome.org/content/detail/R-HSA-177929', 'support': 'Curated human receptor signaling pathway with EGFR as the receptor. EGFR belongs to contextual drug-target biology, not the ranked top-15 LUAD candidates.', 'retrieved': '2026-11-02'}, 'hpo_cough': {'title': 'Cough (HP:0012735)', 'author': 'Human Phenotype Ontology Consortium', 'year': 'n.d.', 'url': 'https://hpo.jax.org/browse/term/HP:0012735', 'verification_url': 'https://raw.githubusercontent.com/obophenotype/human-phenotype-ontology/master/hp.obo', 'corroborating_url': 'https://www.ncbi.nlm.nih.gov/medgen/41325', 'support': 'Identifier and label verified directly in the official HPO ontology on the retrieval date. HPO standardizes the phenotype term; NCI supplies disease-to-symptom evidence.', 'retrieved': '2026-11-02'}, 'hpo_dyspnea': {'title': 'Dyspnea (HP:0002094)', 'author': 'Human Phenotype Ontology Consortium', 'year': 'n.d.', 'url': 'https://hpo.jax.org/browse/term/HP:0002094', 'verification_url': 'https://raw.githubusercontent.com/obophenotype/human-phenotype-ontology/master/hp.obo', 'corroborating_url': 'https://www.ncbi.nlm.nih.gov/medgen/3938', 'support': 'Identifier and label verified directly in the official HPO ontology on the retrieval date. NCI descriptions of breathing difficulty are harmonized to this term; no symptom frequency is inferred.', 'retrieved': '2026-11-02'}, 'clinvar_egfr': {'title': 'NM_005228.5(EGFR):c.2573T>G (p.Leu858Arg), Variation ID 16609', 'author': 'National Center for Biotechnology Information', 'year': 'n.d.', 'url': 'https://www.ncbi.nlm.nih.gov/clinvar/variation/16609/', 'luad_record_url': 'https://www.ncbi.nlm.nih.gov/clinvar/RCV000418019/', 'support': 'Verifies EGFR L858R, transcript NM_005228.5:c.2573T>G, variant identifier 16609 and gene association. The LUAD record documents the disease context. This is an external contextual variant; it is not asserted to occur in this project cohort. Aggregate germline, drug-response, clinical-impact and somatic-oncogenicity classifications must not be conflated.', 'retrieved': '2026-11-02'}, 'hgnc': {'title': 'HGNC gene symbol reports and REST API', 'author': 'HUGO Gene Nomenclature Committee', 'year': 'n.d.', 'url': 'https://www.genenames.org/', 'api_url_template': 'https://rest.genenames.org/fetch/symbol/{symbol}', 'support': 'Official symbol-specific records retrieved directly for the 15 ranked candidates plus EGFR and TOP1. All 17 returned one record with approved symbol, HGNC, Ensembl, NCBI Gene and UniProt identifiers.', 'retrieved': '2026-11-02'}, 'module2': {'title': 'Module 2: LUAD integrated multi-omics results', 'author': 'Kanaparthy, A.', 'year': 2026, 'url': 'https://github.com/Ajaykanaparthy/biot6900/blob/main/module_2/data/luad_integrated_gene_level.csv', 'retrieved': '2026-11-02', 'support': 'Existing project results; source file read directly, not regenerated.'}, 'module3': {'title': 'Module 3: LUAD tractability predictions and target ranking', 'author': 'Kanaparthy, A.', 'year': 2026, 'url': 'https://github.com/Ajaykanaparthy/biot6900/blob/main/module_3/targets_luad_w3.csv', 'retrieved': '2026-11-02', 'support': 'Existing project results; source file read directly, not regenerated.'}, 'module4': {'title': 'Module 4: Verified structural pocket summary', 'author': 'Kanaparthy, A.', 'year': 2026, 'url': 'https://github.com/Ajaykanaparthy/biot6900/blob/main/module_4/results/verified_structure_summary.csv', 'retrieved': '2026-11-02', 'support': 'Existing project results; source file read directly, not regenerated.'}}, 'gene_identifiers': {'ACTN2': {'ensembl': 'ENSG00000077522', 'hgnc_id': 'HGNC:164', 'ncbi_gene_id': '88', 'uniprot_id': 'P35609', 'source': 'https://rest.genenames.org/fetch/symbol/ACTN2', 'retrieved': '2026-11-02'}, 'ZNF536': {'ensembl': 'ENSG00000198597', 'hgnc_id': 'HGNC:29025', 'ncbi_gene_id': '9745', 'uniprot_id': 'O15090', 'source': 'https://rest.genenames.org/fetch/symbol/ZNF536', 'retrieved': '2026-11-02'}, 'COL6A6': {'ensembl': 'ENSG00000206384', 'hgnc_id': 'HGNC:27023', 'ncbi_gene_id': '131873', 'uniprot_id': 'A6NMZ7', 'source': 'https://rest.genenames.org/fetch/symbol/COL6A6', 'retrieved': '2026-11-02'}, 'TNR': {'ensembl': 'ENSG00000116147', 'hgnc_id': 'HGNC:11953', 'ncbi_gene_id': '7143', 'uniprot_id': 'Q92752', 'source': 'https://rest.genenames.org/fetch/symbol/TNR', 'retrieved': '2026-11-02'}, 'SCN7A': {'ensembl': 'ENSG00000136546', 'hgnc_id': 'HGNC:10594', 'ncbi_gene_id': '6332', 'uniprot_id': 'Q01118', 'source': 'https://rest.genenames.org/fetch/symbol/SCN7A', 'retrieved': '2026-11-02'}, 'ABCA8': {'ensembl': 'ENSG00000141338', 'hgnc_id': 'HGNC:38', 'ncbi_gene_id': '10351', 'uniprot_id': 'O94911', 'source': 'https://rest.genenames.org/fetch/symbol/ABCA8', 'retrieved': '2026-11-02'}, 'ADAMTS16': {'ensembl': 'ENSG00000145536', 'hgnc_id': 'HGNC:17108', 'ncbi_gene_id': '170690', 'uniprot_id': 'Q8TE57', 'source': 'https://rest.genenames.org/fetch/symbol/ADAMTS16', 'retrieved': '2026-11-02'}, 'TNXB': {'ensembl': 'ENSG00000168477', 'hgnc_id': 'HGNC:11976', 'ncbi_gene_id': '7148', 'uniprot_id': 'P22105', 'source': 'https://rest.genenames.org/fetch/symbol/TNXB', 'retrieved': '2026-11-02'}, 'NOS1': {'ensembl': 'ENSG00000089250', 'hgnc_id': 'HGNC:7872', 'ncbi_gene_id': '4842', 'uniprot_id': 'P29475', 'source': 'https://rest.genenames.org/fetch/symbol/NOS1', 'retrieved': '2026-11-02'}, 'MYH2': {'ensembl': 'ENSG00000125414', 'hgnc_id': 'HGNC:7572', 'ncbi_gene_id': '4620', 'uniprot_id': 'Q9UKX2', 'source': 'https://rest.genenames.org/fetch/symbol/MYH2', 'retrieved': '2026-11-02'}, 'MASP1': {'ensembl': 'ENSG00000127241', 'hgnc_id': 'HGNC:6901', 'ncbi_gene_id': '5648', 'uniprot_id': 'P48740', 'source': 'https://rest.genenames.org/fetch/symbol/MASP1', 'retrieved': '2026-11-02'}, 'ERICH3': {'ensembl': 'ENSG00000178965', 'hgnc_id': 'HGNC:25346', 'ncbi_gene_id': '127254', 'uniprot_id': 'Q5RHP9', 'source': 'https://rest.genenames.org/fetch/symbol/ERICH3', 'retrieved': '2026-11-02'}, 'COL11A1': {'ensembl': 'ENSG00000060718', 'hgnc_id': 'HGNC:2186', 'ncbi_gene_id': '1301', 'uniprot_id': 'P12107', 'source': 'https://rest.genenames.org/fetch/symbol/COL11A1', 'retrieved': '2026-11-02'}, 'SEMA5A': {'ensembl': 'ENSG00000112902', 'hgnc_id': 'HGNC:10736', 'ncbi_gene_id': '9037', 'uniprot_id': 'Q13591', 'source': 'https://rest.genenames.org/fetch/symbol/SEMA5A', 'retrieved': '2026-11-02'}, 'GDF10': {'ensembl': 'ENSG00000266524', 'hgnc_id': 'HGNC:4215', 'ncbi_gene_id': '2662', 'uniprot_id': 'P55107', 'source': 'https://rest.genenames.org/fetch/symbol/GDF10', 'retrieved': '2026-11-02'}, 'EGFR': {'ensembl': 'ENSG00000146648', 'hgnc_id': 'HGNC:3236', 'ncbi_gene_id': '1956', 'uniprot_id': 'P00533', 'source': 'https://rest.genenames.org/fetch/symbol/EGFR', 'retrieved': '2026-11-02'}, 'TOP1': {'ensembl': 'ENSG00000198900', 'hgnc_id': 'HGNC:11986', 'ncbi_gene_id': '7150', 'uniprot_id': 'P11387', 'source': 'https://rest.genenames.org/fetch/symbol/TOP1', 'retrieved': '2026-11-02'}}, 'ecm_memberships': [{'gene': 'COL6A6', 'pathway_id': 'R-HSA-1474244', 'membership_url': 'https://reactome.org/content/detail/R-HSA-2127489', 'species': 'Homo sapiens', 'retrieved': '2026-11-02'}, {'gene': 'COL11A1', 'pathway_id': 'R-HSA-1474244', 'membership_url': 'https://reactome.org/content/detail/R-HSA-2152278', 'species': 'Homo sapiens', 'retrieved': '2026-11-02'}, {'gene': 'TNXB', 'pathway_id': 'R-HSA-1474244', 'membership_url': 'https://reactome.org/content/detail/R-HSA-2672026', 'species': 'Homo sapiens', 'retrieved': '2026-11-02'}, {'gene': 'TNR', 'pathway_id': 'R-HSA-1474244', 'membership_url': 'https://reactome.org/content/detail/R-HSA-373642', 'species': 'Homo sapiens', 'retrieved': '2026-11-02'}], 'variant': {'name': 'EGFR L858R', 'gene': 'EGFR', 'clinvar_id': '16609', 'hgvs': 'NM_005228.5:c.2573T>G', 'protein_change': 'p.Leu858Arg', 'dbsnp_id': 'rs121434568', 'chromosome': '7', 'position_grch38': 55191822, 'origin': 'external somatic LUAD context; not a cohort call'}, 'interpretation_notes': ['Use missing GWAS values; do not substitute RNA p-values.', 'TOP1-associated-with-SCLC means an approved therapeutic target context; not causal gene or patient mutation.', 'The five-edge topotecan-to-LUAD path via SCLC and cough/dyspnea is a weak hypothesis generated from shared nonspecific clinical features; it does not establish LUAD efficacy.', 'Confidence refers to support for the specific stored edge, not certainty of a biological causal mechanism or a clinical prediction.']}
SOURCES = CATALOG["SOURCES"]
GENE_IDS = CATALOG["gene_identifiers"]
for row in cohort.to_dict("records"):
    assert row["ensembl"] == GENE_IDS[row["gene"]]["ensembl"]
    assert row["accession"] == GENE_IDS[row["gene"]]["uniprot_id"]

PHENOTYPES = ["Cough", "Dyspnea"]
PHENOTYPE_IDS = {"Cough": "HP:0012735", "Dyspnea": "HP:0002094"}
PATHWAYS = ["Extracellular matrix organization", "Lectin pathway of complement activation", "Signaling by EGFR"]
PATHWAY_IDS = dict(zip(PATHWAYS, ["R-HSA-1474244", "R-HSA-166662", "R-HSA-177929"]))
GENE_IN_PATHWAY = {PATHWAYS[0]: ["COL6A6", "COL11A1", "TNXB", "TNR"],
                   PATHWAYS[1]: ["MASP1"], PATHWAYS[2]: ["EGFR"]}
DRUGS = [("Osimertinib", "EGFR", "FDA-approved in specified EGFR-mutant NSCLC settings; 2024 label")]
VARIANTS = [("EGFR L858R", "EGFR")]
RELATED = {"name": "Small-cell lung carcinoma", "shares": PHENOTYPES,
           "genes": ["TOP1"],
           "drugs": [("Topotecan", "TOP1", "FDA-approved for specified recurrent platinum-sensitive SCLC; 2019 label")]}
PATIENTS = []
assert patient_metadata["age_is_simulated"].all() and patient_metadata["phenotype_is_simulated"].all()
print(f"Read {len(pgx)} Module 5 CYP2C19 records as pharmacogenomic context only; none is a LUAD driver call.")
print(f"Patient layer omitted: {len(patient_metadata)} Module 6 genotypes are real, but ages/outcomes are simulated.")

# Confidence categories support only the exact stored assertion, not clinical efficacy.
CONFIDENCE_CRITERIA = {
    "high": "Direct curated identifier/pathway assertion or regulatory drug mechanism/indication.",
    "moderate": "Measured observational association in one cohort, or general clinical symptom description.",
    "low": "An indirect hypothesis without direct experimental support; not used as an asserted molecular edge."
}
NODES, EDGES = {}, []
def clean(value):
    if value is None or (isinstance(value, (float, np.floating)) and not np.isfinite(value)):
        return None
    return value.item() if isinstance(value, np.generic) else value

def add_node(name, kind, **props):
    if name in NODES:
        assert NODES[name]["kind"] == kind, "Identifier collision between node types"
    NODES.setdefault(name, {"kind": kind, "project": PROJECT}).update({k: clean(v) for k, v in props.items()})

def add_edge(src, rel, dst, *, source_id, confidence, evidence_type, evidence_note, **props):
    assert src in NODES and dst in NODES
    assert confidence in CONFIDENCE_CRITERIA and source_id in SOURCES
    EDGES.append(dict(src=src, rel=rel, dst=dst, project=PROJECT, source_id=source_id,
                      source=SOURCES[source_id]["title"], source_url=SOURCES[source_id]["url"],
                      confidence=confidence, confidence_basis=CONFIDENCE_CRITERIA[confidence],
                      evidence_type=evidence_type, evidence_note=evidence_note,
                      verified_on="2026-11-02", **{k: clean(v) for k, v in props.items()}))

add_node(DISEASE, "disease", source_url=SOURCES["nci_nsclc"]["url"], scope="NSCLC adenocarcinoma histology")
add_node(RELATED["name"], "disease", source_url=SOURCES["nci_sclc"]["url"])
for row in cohort.to_dict("records"):
    gene = row["gene"]
    ids = GENE_IDS[gene]
    add_node(gene, "gene", **ids, ranked_top15=True, evidence_origin="measured LUAD cohort plus model-derived tractability",
             gwas_p=None, rna_lfc=row["rna_lfc"], rna_p=row["rna_p"], prot_lfc=row["prot_lfc"],
             prot_p=row["prot_p"], rna_n_pairs=row["rna_n_pairs"], prot_n_pairs=row["prot_n_pairs"],
             mutation_freq=row["mutation_freq"], concordant=row["concordant"], multiomics_score=row["score"],
             p_sm=row["p_sm"], p_ab=row["p_ab"], tractability_type="uncalibrated model priors, not binding evidence",
             module2_rank=row["module2_rank"], module3_rank=row["module3_rank"],
             embedding_truncated=row["embedding_truncated"])
    add_edge(gene, "ASSOCIATED_WITH", DISEASE, source_id="module2", confidence="moderate",
             evidence_type="measured_cohort_association",
             evidence_note="Paired tumor-normal multi-omics and mutation recurrence; association does not prove dependency.",
             rna_p=row["rna_p"], prot_p=row["prot_p"], rna_q_bh=row["rna_q_bh"], prot_q_bh=row["prot_q_bh"],
             rna_lfc=row["rna_lfc"], prot_lfc=row["prot_lfc"], mutation_freq=row["mutation_freq"],
             gwas_p=None, source_file="module_2/data/luad_integrated_gene_level.csv")

# Context genes enable established pharmacology and are excluded from cohort rankings.
for gene in ["EGFR", "TOP1"]:
    add_node(gene, "gene", **GENE_IDS[gene], ranked_top15=False,
             evidence_origin="external therapeutic context; not a ranked cohort candidate", gwas_p=None)
add_edge("EGFR", "ASSOCIATED_WITH", DISEASE, source_id="fda_osimertinib", confidence="high",
         evidence_type="regulatory_biomarker_context",
         evidence_note="EGFR-mutated NSCLC includes LUAD; not every LUAD has an eligible EGFR mutation.")
add_edge("TOP1", "ASSOCIATED_WITH", RELATED["name"], source_id="fda_topotecan", confidence="high",
         evidence_type="regulatory_therapeutic_target_context",
         evidence_note="Target of an approved SCLC medicine; this does not assert that TOP1 is an SCLC causal driver.")
for phenotype in PHENOTYPES:
    hpo_key = "hpo_cough" if phenotype == "Cough" else "hpo_dyspnea"
    add_node(phenotype, "phenotype", ontology_id=PHENOTYPE_IDS[phenotype], source_url=SOURCES[hpo_key]["url"])
    for disease, source_id in [(DISEASE, "nci_nsclc"), (RELATED["name"], "nci_sclc")]:
        add_edge(disease, "HAS_PHENOTYPE", phenotype, source_id=source_id, confidence="moderate",
                 evidence_type="clinical_description",
                 evidence_note="NCI clinical description harmonized to HPO; nonspecific symptom, no cohort prevalence inferred.",
                 ontology_id=PHENOTYPE_IDS[phenotype], ontology_source_url=SOURCES[hpo_key]["url"])
membership_urls = {row["gene"]: row["membership_url"] for row in CATALOG["ecm_memberships"]}
for pathway, genes in GENE_IN_PATHWAY.items():
    source_id = {PATHWAYS[0]: "reactome_ecm", PATHWAYS[1]: "reactome_lectin", PATHWAYS[2]: "reactome_egfr"}[pathway]
    add_node(pathway, "pathway", reactome_id=PATHWAY_IDS[pathway], source_url=SOURCES[source_id]["url"])
    for gene in genes:
        add_edge(gene, "PARTICIPATES_IN", pathway, source_id=source_id, confidence="high",
                 evidence_type="curated_human_pathway_membership",
                 evidence_note="Human Reactome pathway membership; no LUAD-specific activation or causal effect inferred.",
                 reactome_id=PATHWAY_IDS[pathway], membership_source_url=membership_urls.get(gene, SOURCES[source_id]["url"]))
for drug, gene, status in DRUGS + RELATED["drugs"]:
    source_id = "fda_osimertinib" if drug == "Osimertinib" else "fda_topotecan"
    add_node(drug, "drug", status=status, source_url=SOURCES[source_id]["url"])
    add_edge(drug, "TARGETS", gene, source_id=source_id, confidence="high",
             evidence_type="regulatory_drug_mechanism", evidence_note=SOURCES[source_id]["support"])
for variant, gene in VARIANTS:
    add_node(variant, "variant", **{k: v for k, v in CATALOG["variant"].items() if k not in {"name", "gene"}},
             source_url=SOURCES["clinvar_egfr"]["url"])
    add_edge(variant, "OCCURS_IN", gene, source_id="clinvar_egfr", confidence="high",
             evidence_type="verified_external_variant_gene_mapping",
             evidence_note="EGFR L858R external LUAD context; no assertion this variant was observed in the project cohort.",
             clinvar_id="16609", hgvs="NM_005228.5:c.2573T>G")
for row in structure.to_dict("records"):
    if row["gene"] in NODES:
        NODES[row["gene"]].update(structure_druggability_score=clean(row["best_pocket_druggability_score"]),
                                 structure_evidence_type="computational pocket, not measured binding",
                                 structure_source_url=SOURCES["module4"]["url"])

assert len({(e["src"], e["rel"], e["dst"]) for e in EDGES}) == len(EDGES)
assert all(e["source_url"] and e["confidence"] and e["confidence_basis"] for e in EDGES)
assert all(NODES[gene]["gwas_p"] is None for gene in EXPECTED_GENES)
print("nodes:", len(NODES), " edges:", len(EDGES), " disease:", DISEASE)
display(cohort[["gene", "ensembl", "accession", "score", "p_sm", "p_ab", "module2_rank", "module3_rank"]])


Read 33 Module 5 CYP2C19 records as pharmacogenomic context only; none is a LUAD driver call.
Patient layer omitted: 2504 Module 6 genotypes are real, but ages/outcomes are simulated.
nodes: 27  edges: 30  disease: Lung adenocarcinoma (LUAD)


,gene,ensembl,accession,score,p_sm,p_ab,module2_rank,module3_rank
0,ACTN2,ENSG00000077522,P35609,0.992157,0.219997,0.000366,1,2
1,ZNF536,ENSG00000198597,O15090,0.987570,0.002995,0.000401,2,7
2,COL6A6,ENSG00000206384,A6NMZ7,0.985918,0.280420,0.030515,3,1
3,TNR,ENSG00000116147,Q92752,0.985005,0.140365,0.235683,4,4
4,SCN7A,ENSG00000136546,Q01118,0.982367,0.644801,0.000398,5,3
5,ABCA8,ENSG00000141338,O94911,0.975782,0.180463,0.004178,6,6
6,ADAMTS16,ENSG00000145536,Q8TE57,0.972650,0.098899,0.099197,7,10
7,TNXB,ENSG00000168477,P22105,0.972181,0.043937,0.278933,8,5
8,NOS1,ENSG00000089250,P29475,0.971811,0.204068,0.000746,9,9
9,MYH2,ENSG00000125414,Q9UKX2,0.970849,0.046894,0.003107,10,12


## 2 · Load into Neo4j

Provided — no edits needed. Creates a uniqueness constraint, then loads your
nodes and relationships with batched `UNWIND` + `MERGE`.

In [4]:
# MERGE only; preserve every pre-existing database node and relationship.
existing_node_ids = {r["id"] for r in run_query("MATCH (n) RETURN elementId(n) AS id")}
existing_edge_ids = {r["id"] for r in run_query("MATCH ()-[r]->() RETURN elementId(r) AS id")}
run_query("CREATE CONSTRAINT entity_name IF NOT EXISTS "
          "FOR (n:Entity) REQUIRE n.name IS UNIQUE")
KIND_LABEL = {"gene": "Gene", "disease": "Disease", "phenotype": "Phenotype", "pathway": "Pathway",
              "drug": "Drug", "variant": "Variant", "patient": "Patient"}
for kind, label in KIND_LABEL.items():
    rows = [dict(name=name, **data) for name, data in NODES.items() if data["kind"] == kind]
    if rows:
        run_query(f"UNWIND $rows AS row MERGE (e:Entity {{name: row.name}}) "
                  f"SET e:{label}:Module7LUAD SET e += row", rows=rows)
by_rel = defaultdict(list)
for edge in EDGES:
    by_rel[edge["rel"]].append({"src": edge["src"], "dst": edge["dst"],
                               "props": {k: v for k, v in edge.items() if k not in {"src", "dst", "rel"}}})
for rel, rows in by_rel.items():
    assert rel in {"ASSOCIATED_WITH", "HAS_PHENOTYPE", "PARTICIPATES_IN", "TARGETS", "OCCURS_IN"}
    run_query(f"""UNWIND $rows AS row
                  MATCH (a:Entity {{name: row.src}})
                  MATCH (b:Entity {{name: row.dst}})
                  MERGE (a)-[r:{rel}]->(b) SET r += row.props""", rows=rows)

# Scope assignment counts instead of counting unrelated pre-existing examples.
db_nodes = run_query("MATCH (n:Module7LUAD) RETURN n.name AS name, n.kind AS kind")
db_edges = run_query("MATCH (a:Module7LUAD)-[r]->(b:Module7LUAD) WHERE r.project = $project "
                     "RETURN a.name AS src, type(r) AS rel, b.name AS dst, properties(r) AS props", project=PROJECT)
assert {r["name"] for r in db_nodes} == set(NODES)
assert {(e["src"], e["rel"], e["dst"]) for e in db_edges} == {(e["src"], e["rel"], e["dst"]) for e in EDGES}
assert all(e["props"].get("source_url") and e["props"].get("confidence") for e in db_edges)
assert existing_node_ids <= {r["id"] for r in run_query("MATCH (n) RETURN elementId(n) AS id")}
assert existing_edge_ids <= {r["id"] for r in run_query("MATCH ()-[r]->() RETURN elementId(r) AS id")}
summary = dict(node_count=len(db_nodes), edge_count=len(db_edges),
               node_counts=dict(Counter(r["kind"] for r in db_nodes)),
               relationship_counts=dict(Counter(r["rel"] for r in db_edges)),
               preexisting_nodes_preserved=len(existing_node_ids),
               preexisting_relationships_preserved=len(existing_edge_ids), connected_to_neo4j=True)
print("LUAD assignment node counts:", summary["node_counts"])
print("LUAD assignment relationship counts:", summary["relationship_counts"])
print("Total assignment nodes:", summary["node_count"], "relationships:", summary["edge_count"])
print("Pre-existing database records preserved; no reset/delete was performed.")


LUAD assignment node counts: {'gene': 17, 'disease': 2, 'phenotype': 2, 'pathway': 3, 'drug': 2, 'variant': 1}
LUAD assignment relationship counts: {'ASSOCIATED_WITH': 17, 'PARTICIPATES_IN': 6, 'HAS_PHENOTYPE': 4, 'TARGETS': 2, 'OCCURS_IN': 1}
Total assignment nodes: 27 relationships: 30
Pre-existing database records preserved; no reset/delete was performed.


## 3 · A quick look

Provided. Open your instance in the Aura console and run
`MATCH (n) RETURN n LIMIT 100` to see the graph you just loaded, or use §7 to draw it.

## 4 · Query it — the opening question (Cypher)

Provided. These answer the opening question for whatever `DISEASE` you set in §1.

> **TODO (report):** after running, note which drugs come back, and for the
> repurposing query, which related disease and shared symptom produced each hit.

In [5]:
# 2 hops: drugs acting on genes associated with your disease
two_hop_results = run_query("""
    MATCH (d:Drug)-[:TARGETS]->(g:Gene)-[:ASSOCIATED_WITH]->(dis:Disease)
    WHERE dis.name = $disease
    RETURN d.name AS drug, g.name AS gene, d.status AS status
    ORDER BY drug
""", disease=DISEASE)

display(pd.DataFrame(two_hop_results))
two_hop_evidence = run_query("""
    MATCH (d:Drug)-[t:TARGETS]->(g:Gene)-[a:ASSOCIATED_WITH]->(dis:Disease)
    WHERE dis.name = $disease
    RETURN d.name AS drug, g.name AS gene, d.status AS status,
           t.source_url AS target_source, t.confidence AS target_confidence,
           a.source_url AS disease_source, a.evidence_type AS disease_evidence_type
    ORDER BY drug
""", disease=DISEASE)
assert two_hop_results
display(pd.DataFrame(two_hop_evidence))
print("EGFR is external treatment context, not one of the ranked 15. The label applies to eligible EGFR-mutant NSCLC settings.")


,drug,gene,status
0,Osimertinib,EGFR,FDA-approved in specified EGFR-mutant NSCLC se...


,drug,gene,status,target_source,target_confidence,disease_source,disease_evidence_type
0,Osimertinib,EGFR,FDA-approved in specified EGFR-mutant NSCLC se...,https://www.accessdata.fda.gov/drugsatfda_docs...,high,https://www.accessdata.fda.gov/drugsatfda_docs...,regulatory_biomarker_context


EGFR is external treatment context, not one of the ranked 15. The label applies to eligible EGFR-mutant NSCLC settings.


In [6]:
# Five nodes / four relationships: drugs on genes of diseases that SHARE A SYMPTOM with your disease
repurposing_results = run_query("""
    MATCH (d:Drug)-[:TARGETS]->(g:Gene)-[:ASSOCIATED_WITH]->(other:Disease)
          -[:HAS_PHENOTYPE]->(p:Phenotype)<-[:HAS_PHENOTYPE]-(ad:Disease)
    WHERE ad.name = $disease AND other <> ad
    RETURN d.name AS repurposing_candidate, g.name AS gene,
           other.name AS via_disease, collect(DISTINCT p.name) AS shared_symptoms
    ORDER BY repurposing_candidate
""", disease=DISEASE)

display(pd.DataFrame(repurposing_results))
path_evidence = run_query("""
    MATCH (d:Drug)-[t:TARGETS]->(g:Gene)-[a:ASSOCIATED_WITH]->(other:Disease)
          -[s:HAS_PHENOTYPE]->(p:Phenotype)<-[h:HAS_PHENOTYPE]-(ad:Disease)
    WHERE ad.name = $disease AND other <> ad
    RETURN [d.name, g.name, other.name, p.name, ad.name] AS nodes,
           [type(t), type(a), type(s), type(h)] AS relationships,
           [t.source_url, a.source_url, s.source_url, h.source_url] AS source_urls,
           [t.confidence, a.confidence, s.confidence, h.confidence] AS confidence,
           a.evidence_type AS gene_disease_evidence_type
    ORDER BY p.name, d.name
""", disease=DISEASE)
assert path_evidence
for path in path_evidence:
    d, g, other, symptom, luad = path["nodes"]
    print(f"{d} -[:TARGETS]-> {g} -[:ASSOCIATED_WITH]-> {other} "
          f"-[:HAS_PHENOTYPE]-> {symptom} <-[:HAS_PHENOTYPE]- {luad}")
print("TOP1-SCLC is therapeutic-target context. Shared cough/dyspnea is nonspecific; this path does not establish topotecan efficacy in LUAD.")
print("Test hypothesis with exposure-matched LUAD and SCLC cell/organoid panels, normal lung controls, TOP1 perturbation, DNA-damage and viability assays.")


,repurposing_candidate,gene,via_disease,shared_symptoms
0,Topotecan,TOP1,Small-cell lung carcinoma,"[Cough, Dyspnea]"


Topotecan -[:TARGETS]-> TOP1 -[:ASSOCIATED_WITH]-> Small-cell lung carcinoma -[:HAS_PHENOTYPE]-> Cough <-[:HAS_PHENOTYPE]- Lung adenocarcinoma (LUAD)
Topotecan -[:TARGETS]-> TOP1 -[:ASSOCIATED_WITH]-> Small-cell lung carcinoma -[:HAS_PHENOTYPE]-> Dyspnea <-[:HAS_PHENOTYPE]- Lung adenocarcinoma (LUAD)
TOP1-SCLC is therapeutic-target context. Shared cough/dyspnea is nonspecific; this path does not establish topotecan efficacy in LUAD.
Test hypothesis with exposure-matched LUAD and SCLC cell/organoid panels, normal lung controls, TOP1 perturbation, DNA-damage and viability assays.


## 5 · Biomarker prioritization (centrality)

Provided. PageRank fused with your gene evidence.

> **TODO (report):** which genes rank highest? Does that match the biology you
> expect for your disease? Flag any non-obvious hit.

In [7]:
# Preserve the professor's directed NetworkX PageRank calculation.
G = nx.DiGraph()
for name, data in NODES.items():
    G.add_node(name, **data)
for edge in EDGES:
    G.add_edge(edge["src"], edge["dst"], **{k: v for k, v in edge.items() if k not in {"src", "dst"}})
pr = nx.pagerank(G)
biomarkers = cohort.copy().rename(columns={"score": "multiomics_score"})
biomarkers["pagerank"] = biomarkers["gene"].map(pr)

# Within the same 15 candidates, percentile normalization preserves source ordering.
# If a component is constant, use neutral 0.5 rather than implying discrimination.
def normalized_percentile(values):
    return pd.Series(0.5, index=values.index) if np.isclose(values.max(), values.min(), rtol=0, atol=1e-12) else values.rank(method="average", pct=True)
biomarkers["centrality_pct"] = normalized_percentile(biomarkers["pagerank"])
biomarkers["multiomics_pct"] = normalized_percentile(biomarkers["multiomics_score"])
biomarkers["tractability_prior"] = biomarkers[["p_sm", "p_ab"]].max(axis=1)
biomarkers["tractability_pct"] = normalized_percentile(biomarkers["tractability_prior"])
biomarkers["combined_priority"] = (0.40 * biomarkers["centrality_pct"] +
                                    0.40 * biomarkers["multiomics_pct"] +
                                    0.20 * biomarkers["tractability_pct"]).round(12)
# Remove machine-precision noise before ranking; mathematically equal scores stay tied.
def evidence_limitations(row):
    notes = ["observational association; no GWAS or dependency evidence", "p_sm/p_ab are uncalibrated model priors"]
    if not row["concordant"]:
        notes.append("opposite RNA/protein directions")
    if row["prot_n_pairs"] < 50:
        notes.append(f"only {int(row['prot_n_pairs'])} valid protein pairs")
    if row["embedding_truncated"]:
        notes.append("protein embedding truncated")
    if row["gene"] in set(structure["gene"]):
        notes.append("structural pocket is computational, not binding validation")
    return "; ".join(notes)
biomarkers["limitations"] = biomarkers.apply(evidence_limitations, axis=1)
biomarkers = biomarkers.sort_values(["combined_priority", "module2_rank"], ascending=[False, True]).reset_index(drop=True)
biomarkers["combined_rank"] = biomarkers["combined_priority"].rank(method="min", ascending=False).astype(int)
summary.update(directed_pagerank_tied=bool(np.ptp(biomarkers["pagerank"]) < 1e-12),
               rank_corr_module2=float(biomarkers["combined_rank"].corr(biomarkers["module2_rank"], method="spearman")),
               rank_corr_module3=float(biomarkers["combined_rank"].corr(biomarkers["module3_rank"], method="spearman")),
               weighting={"directed_pagerank": 0.4, "module2_multiomics": 0.4, "module3_tractability": 0.2},
               top_biomarkers=biomarkers.head(5)["gene"].tolist())
display(biomarkers[["combined_rank", "gene", "pagerank", "multiomics_score", "p_sm", "p_ab", "combined_priority", "module2_rank", "module3_rank", "limitations"]])
print("Weights: 40% centrality + 40% Module 2 multi-omics + 20% max(p_sm,p_ab) modality prior; teaching choices, not fitted clinical weights.")
if summary["directed_pagerank_tied"]:
    print("All 15 cohort genes tie in directed PageRank because edges leave genes and no drugs/variants point into them. Its normalized contribution is neutral 0.5.")
print("Fused-vs-Module2 Spearman:", round(summary["rank_corr_module2"], 3),
      "Fused-vs-Module3:", round(summary["rank_corr_module3"], 3))
print("Contextual EGFR/TOP1 excluded from this ranking. No GWAS p-values were supplied or inferred.")


,combined_rank,gene,pagerank,multiomics_score,p_sm,p_ab,combined_priority,module2_rank,module3_rank,limitations
0,1,ACTN2,0.016889,0.992157,0.219997,0.000366,0.720000,1,2,observational association; no GWAS or dependen...
1,1,COL6A6,0.016889,0.985918,0.280420,0.030515,0.720000,3,1,observational association; no GWAS or dependen...
2,3,SCN7A,0.016889,0.982367,0.644801,0.000398,0.693333,5,3,observational association; no GWAS or dependen...
3,4,TNR,0.016889,0.985005,0.140365,0.235683,0.653333,4,4,observational association; no GWAS or dependen...
4,5,ZNF536,0.016889,0.987570,0.002995,0.000401,0.586667,2,7,observational association; no GWAS or dependen...
5,6,TNXB,0.016889,0.972181,0.043937,0.278933,0.573333,8,5,observational association; no GWAS or dependen...
6,7,ABCA8,0.016889,0.975782,0.180463,0.004178,0.560000,6,6,observational association; no GWAS or dependen...
7,8,MASP1,0.016889,0.970233,0.604960,0.549767,0.520000,11,8,observational association; no GWAS or dependen...
8,9,ADAMTS16,0.016889,0.972650,0.098899,0.099197,0.493333,7,10,observational association; no GWAS or dependen...
9,9,NOS1,0.016889,0.971811,0.204068,0.000746,0.493333,9,9,observational association; no GWAS or dependen...


Weights: 40% centrality + 40% Module 2 multi-omics + 20% max(p_sm,p_ab) modality prior; teaching choices, not fitted clinical weights.
All 15 cohort genes tie in directed PageRank because edges leave genes and no drugs/variants point into them. Its normalized contribution is neutral 0.5.
Fused-vs-Module2 Spearman: 0.913 Fused-vs-Module3: 0.987
Contextual EGFR/TOP1 excluded from this ranking. No GWAS p-values were supplied or inferred.


## 6 · Communities

Provided. Modularity clustering groups densely connected nodes.

> **TODO (report):** pick ONE community and read it as a subtype or mechanism
> hypothesis.

In [8]:
from networkx.algorithms.community import greedy_modularity_communities
comms = greedy_modularity_communities(G.to_undirected())
print(f"{len(comms)} communities:")
community_rows = []
for i, community in enumerate(comms):
    print(f"  {i}: {sorted(community)}")
    for name in sorted(community):
        community_rows.append(dict(community=i, name=name, kind=NODES[name]["kind"]))
selected_community = next(c for c in comms if PATHWAYS[0] in c)
summary.update(community_count=len(comms), selected_community=sorted(selected_community),
               communities=[sorted(c) for c in comms])
print("Selected ECM community:", sorted(selected_community))
print("Matrix genes and their curated pathway suggest tissue-composition/stromal remodeling as a mechanism to test. This is a topology-dependent hypothesis, not a validated patient subtype.")
print("Curation is sparse and unweighted; missing pathway annotations and the disease hub can change membership. Bulk-tissue composition can explain expression changes.")


5 communities:
  0: ['ABCA8', 'ACTN2', 'ADAMTS16', 'ERICH3', 'GDF10', 'Lung adenocarcinoma (LUAD)', 'MYH2', 'NOS1', 'SCN7A', 'SEMA5A', 'ZNF536']
  1: ['COL11A1', 'COL6A6', 'Extracellular matrix organization', 'TNR', 'TNXB']
  2: ['Cough', 'Dyspnea', 'Small-cell lung carcinoma', 'TOP1', 'Topotecan']
  3: ['EGFR', 'EGFR L858R', 'Osimertinib', 'Signaling by EGFR']
  4: ['Lectin pathway of complement activation', 'MASP1']
Selected ECM community: ['COL11A1', 'COL6A6', 'Extracellular matrix organization', 'TNR', 'TNXB']
Matrix genes and their curated pathway suggest tissue-composition/stromal remodeling as a mechanism to test. This is a topology-dependent hypothesis, not a validated patient subtype.
Curation is sparse and unweighted; missing pathway annotations and the disease hub can change membership. Bulk-tissue composition can explain expression changes.


## 7 · Visualize and export

Provided. Draws your graph coloured by kind and exports your node/edge table.
Both the figure and the CSV are graded artifacts — commit them.

In [9]:
import textwrap
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

KIND_COLOR = {"gene": "#D41B2C", "disease": "#3E6F99", "phenotype": "#2FA3D6",
              "pathway": "#00A58C", "drug": "#9A5FC0", "variant": "#FF854F", "patient": "#8A94A3"}
H = G.subgraph([n for n, data in G.nodes(data=True) if data["kind"] != "patient"])
# Undirected projection is used only for display; analysis retains directed edges.
pos = nx.spring_layout(H.to_undirected(), seed=7, k=1.2, iterations=400)
fig, ax = plt.subplots(figsize=(17, 12))
nx.draw_networkx_edges(H, pos, ax=ax, edge_color="#A6AFB8", width=1.0, arrows=True, arrowsize=11,
                       node_size=650, connectionstyle="arc3,rad=0.03")
nx.draw_networkx_nodes(H, pos, ax=ax, node_size=650,
                       node_color=[KIND_COLOR[H.nodes[n]["kind"]] for n in H], edgecolors="white")
labels = {name: "\n".join(textwrap.wrap(name, width=23)) for name in H}
nx.draw_networkx_labels(H, pos, labels=labels, ax=ax, font_size=8,
                        bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=0.8))
handles = [Line2D([0], [0], marker="o", linestyle="", markersize=9, color=color, label=kind.title())
           for kind, color in KIND_COLOR.items() if kind in {H.nodes[n]["kind"] for n in H}]
ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.035), ncol=6, frameon=False)
ax.set_title(f"LUAD knowledge graph: {len(NODES)} nodes, {len(EDGES)} relationships", fontsize=16, color="black")
ax.margins(0.12); ax.axis("off"); fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "my_knowledge_graph.png", dpi=180, bbox_inches="tight", facecolor="white")
plt.show(); plt.close(fig)

# A legible focused view shows actual pharmacology, variant and shared-phenotype paths.
focus_names = [DISEASE, RELATED["name"], "Cough", "Dyspnea", "EGFR", "TOP1", "Osimertinib", "Topotecan", "EGFR L858R"]
F = G.subgraph(focus_names)
focus_pos = {"Osimertinib": (-4, 2.5), "EGFR": (-1, 2.5), DISEASE: (2.5, 2.5),
             "EGFR L858R": (-1, 4.5), "Topotecan": (-4, -2), "TOP1": (-1, -2),
             RELATED["name"]: (2.5, -2), "Cough": (6, 1.8), "Dyspnea": (6, -1.3)}
fig, ax = plt.subplots(figsize=(12, 4.3))
nx.draw_networkx_nodes(F, focus_pos, ax=ax, node_size=1150,
                       node_color=[KIND_COLOR[F.nodes[n]["kind"]] for n in F], edgecolors="white")
nx.draw_networkx_edges(F, focus_pos, ax=ax, edge_color="#6F7780", arrows=True, arrowsize=16,
                       node_size=1400, connectionstyle="arc3,rad=0.04")
focus_labels = {n: "\n".join(textwrap.wrap(n, 19)) for n in F}
# Standard abbreviations keep labels clear; full disease names remain in the exports.
focus_labels[DISEASE] = "LUAD"
focus_labels[RELATED["name"]] = "SCLC"
nx.draw_networkx_labels(F, focus_pos, labels=focus_labels,
                        ax=ax, font_size=16, bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1))
nx.draw_networkx_edge_labels(F, focus_pos, edge_labels={(s, d): a["rel"] for s, d, a in F.edges(data=True)},
                             ax=ax, font_size=11, rotate=False, label_pos=0.51,
                             bbox=dict(facecolor="white", edgecolor="none", alpha=0.8))
ax.legend(handles=[h for h in handles if h.get_label().lower() in {F.nodes[n]["kind"] for n in F}],
          loc="lower center", bbox_to_anchor=(0.5, -0.04), ncol=5, frameon=False, fontsize=12.5)
ax.set_title("LUAD treatment context and shared-symptom hypothesis", fontsize=15, color="black")
ax.text(0.5, -0.15, "LUAD: lung adenocarcinoma; SCLC: small-cell lung carcinoma", transform=ax.transAxes,
        ha="center", fontsize=10, color="black")
ax.margins(0.16); ax.axis("off"); fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "luad_focused_paths.png", dpi=180, bbox_inches="tight", facecolor="white")
plt.show(); plt.close(fig)

node_table = pd.DataFrame([dict(name=name, **data) for name, data in NODES.items()])
edge_table = pd.DataFrame(EDGES)
node_table.to_csv(ARTIFACT_DIR / "nodes.csv", index=False)
edge_table.to_csv(ARTIFACT_DIR / "edges.csv", index=False)
biomarkers.to_csv(ARTIFACT_DIR / "biomarkers.csv", index=False)
pd.DataFrame(community_rows).to_csv(ARTIFACT_DIR / "communities.csv", index=False)
query_rows = []
for i, path in enumerate(path_evidence, 1):
    for hop, (src, dst, rel, url, confidence) in enumerate(zip(path["nodes"][:-1], path["nodes"][1:], path["relationships"], path["source_urls"], path["confidence"]), 1):
        # Last hop is traversed against the stored disease->phenotype arrow.
        query_rows.append(dict(query="shared_phenotype", path_id=i, hop=hop, src=src, dst=dst,
                               rel=rel, traversal="reverse" if hop == 4 else "forward", source_url=url, confidence=confidence))
for row in two_hop_evidence:
    query_rows.append(dict(query="two_hop", drug=row["drug"], gene=row["gene"], status=row["status"],
                           source_url=row["target_source"], disease_source=row["disease_source"],
                           confidence=row["target_confidence"]))
pd.DataFrame(query_rows).to_csv(ARTIFACT_DIR / "query_evidence.csv", index=False)
pd.DataFrame([dict(source_id=k, **v) for k, v in SOURCES.items()]).to_csv(ARTIFACT_DIR / "sources.csv", index=False)
summary.update(two_hop_results=two_hop_results, repurposing_results=repurposing_results, path_evidence=path_evidence,
               confidence_criteria=CONFIDENCE_CRITERIA, patient_layer="Omitted: no observed LUAD ages/outcomes in Module 6",
               pgx_context="CYP2C19 Module 5 variants are not LUAD oncogenic mutations")
(ARTIFACT_DIR / "analysis_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False)+"\n", encoding="utf-8")
assert len(pd.read_csv(ARTIFACT_DIR / "nodes.csv")) == summary["node_count"]
assert len(pd.read_csv(ARTIFACT_DIR / "edges.csv")) == summary["edge_count"]
print("Saved full/focused PNGs, nodes.csv, edges.csv, biomarkers.csv, communities.csv, query_evidence.csv, sources.csv and analysis_summary.json in Module_7.")


C:\Users\kanap\AppData\Local\Temp\ipykernel_43648\3342142327.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close(fig)


Saved full/focused PNGs, nodes.csv, edges.csv, biomarkers.csv, communities.csv, query_evidence.csv, sources.csv and analysis_summary.json in Module_7.


C:\Users\kanap\AppData\Local\Temp\ipykernel_43648\3342142327.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close(fig)


## 8 · Your report (the 40%)

Write a **3–4 page report** answering these four, in your own words:

1. **Schema and evidence** — why these node and edge types, and what evidence you put on the edges.
2. **Top targets** — which genes rank highest by centrality (§5), and does that match the biology you expect? Any non-obvious hit?
3. **A hypothesis from a path** — one repurposing or shared-mechanism hypothesis a path in your graph suggests (§4), and what would test it.
4. **Limitations** — an edge is evidence, not truth. Which of your edges are weak, and how would the answer change with better data? Note where you used (or would use) AI to harmonize data or write queries, and how you checked it.

#### Submit
- This notebook (**Kernel → Restart & Run All** first; credentials in a git-ignored file, not committed).
- `my_knowledge_graph.png`, `nodes.csv`, `edges.csv`.
- Your 3–4 page report (PDF) and a short README (name + your disease).
- Push to your biot6900 repo and post the link on Canvas.

#### Reset (to start over)
```cypher
MATCH (n) DETACH DELETE n      // wipes the database — run deliberately
```


In [10]:
# Generate the report and README from the completed notebook's actual results.
from html import escape as _m7_escape
from pathlib import Path as _m7_Path
from xml.sax.saxutils import quoteattr as _m7_quoteattr
import math as _m7_math
import pandas as _m7_pd
from reportlab.lib import colors as _m7_colors
from reportlab.lib.enums import TA_CENTER as _m7_CENTER, TA_LEFT as _m7_LEFT
from reportlab.lib.pagesizes import letter as _m7_letter
from reportlab.lib.styles import ParagraphStyle as _m7_Style
from reportlab.lib.units import inch as _m7_inch
from reportlab.lib.utils import ImageReader as _m7_ImageReader
from reportlab.platypus import (
    SimpleDocTemplate as _m7_Doc, Paragraph as _m7_Paragraph,
    Spacer as _m7_Spacer, Table as _m7_Table, TableStyle as _m7_TableStyle,
    Image as _m7_Image, PageBreak as _m7_PageBreak, KeepInFrame as _m7_KeepInFrame,
)

_m7_required_columns = {
    "gene", "pagerank", "multiomics_score", "p_sm", "p_ab", "combined_priority",
    "module2_rank", "module3_rank", "combined_rank", "rna_lfc", "prot_lfc",
    "prot_n_pairs", "limitations",
}
if not _m7_required_columns.issubset(biomarkers.columns):
    raise ValueError("The report requires the completed biomarker table.")
if len(biomarkers) != 15 or biomarkers["gene"].nunique() != 15:
    raise ValueError("The report must rank the original 15 distinct LUAD candidates.")
if not two_hop_results or not repurposing_results:
    raise ValueError("Run both Neo4j queries successfully before generating the final report.")
_m7_required_sources = (
    "nci_nsclc", "nci_sclc", "fda_osimertinib", "fda_topotecan",
    "reactome_ecm", "reactome_lectin", "reactome_egfr", "hpo_cough",
    "hpo_dyspnea", "clinvar_egfr", "hgnc", "module2", "module3", "module4",
)
for _m7_key in _m7_required_sources:
    if _m7_key not in SOURCES or not SOURCES[_m7_key].get("title"):
        raise ValueError("Complete the verified source catalog before generating the report.")

def _m7_text(value):
    return _m7_escape(str(value))

def _m7_number(value, decimals=3):
    try:
        return f"{float(value):.{decimals}f}" if _m7_math.isfinite(float(value)) else "unavailable"
    except (TypeError, ValueError):
        return "unavailable"

def _m7_author(source):
    value = source.get("author") or source["title"]
    return "; ".join(map(str, value)) if isinstance(value, (list, tuple)) else str(value)

def _m7_intext_author(source):
    if source.get("citation_author"):
        return str(source["citation_author"])
    author = _m7_author(source)
    if author == "Ajay Kanaparthy":
        return "Kanaparthy"
    return author.split(",", 1)[0] if author.startswith("Kanaparthy,") else author

# Distinguish same-author/same-year sources consistently in APA in-text citations.
_m7_citation_year = {}
_m7_groups = {}
for _m7_key in _m7_required_sources:
    _m7_src = SOURCES[_m7_key]
    _m7_groups.setdefault((_m7_author(_m7_src), str(_m7_src.get("year", "n.d."))), []).append(_m7_key)
for (_m7_author_name, _m7_year), _m7_keys in _m7_groups.items():
    _m7_keys.sort(key=lambda key: str(SOURCES[key]["title"]).lower())
    for _m7_index, _m7_key in enumerate(_m7_keys):
        _m7_suffix = chr(97 + _m7_index) if len(_m7_keys) > 1 else ""
        _m7_citation_year[_m7_key] = _m7_year + ("-" if _m7_year == "n.d." and _m7_suffix else "") + _m7_suffix

def _m7_cite(*keys):
    groups = {}
    for key in keys:
        groups.setdefault(_m7_intext_author(SOURCES[key]), []).append(_m7_citation_year[key])
    return "(" + "; ".join(
        f"{_m7_text(author)}, {_m7_text(', '.join(sorted(set(years))))}"
        for author, years in sorted(groups.items(), key=lambda item: item[0].lower())
    ) + ")"

_m7_styles = {
    "title": _m7_Style("M7Title", fontName="Helvetica-Bold", fontSize=16, leading=19,
                       textColor=_m7_colors.black, spaceAfter=7),
    "subtitle": _m7_Style("M7Subtitle", fontName="Helvetica", fontSize=10, leading=13,
                          textColor=_m7_colors.black, spaceAfter=9),
    "heading": _m7_Style("M7Heading", fontName="Helvetica-Bold", fontSize=12, leading=15,
                         textColor=_m7_colors.black, spaceBefore=8, spaceAfter=6),
    "body": _m7_Style("M7Body", fontName="Helvetica", fontSize=9.5, leading=12.4,
                      textColor=_m7_colors.black, spaceAfter=7, splitLongWords=True),
    "small": _m7_Style("M7Small", fontName="Helvetica", fontSize=8.4, leading=10.6,
                       textColor=_m7_colors.black, spaceAfter=5, splitLongWords=True),
    "table": _m7_Style("M7Table", fontName="Helvetica", fontSize=8.2, leading=10.1,
                       textColor=_m7_colors.black, splitLongWords=True),
    "reference": _m7_Style("M7Reference", fontName="Helvetica", fontSize=8.8, leading=11,
                           textColor=_m7_colors.black, leftIndent=13, firstLineIndent=-13,
                           spaceAfter=8, splitLongWords=True),
}

def _m7_p(value, style="body"):
    return _m7_Paragraph(value, _m7_styles[style])

def _m7_table(rows, widths):
    table = _m7_Table([[_m7_p(str(cell), "table") for cell in row] for row in rows],
                     colWidths=widths, hAlign="LEFT", repeatRows=1)
    table.setStyle(_m7_TableStyle([
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LINEABOVE", (0, 0), (-1, 0), 0.7, _m7_colors.black),
        ("LINEBELOW", (0, 0), (-1, 0), 0.5, _m7_colors.black),
        ("LINEBELOW", (0, -1), (-1, -1), 0.5, _m7_colors.black),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
    ]))
    return table

_m7_output = _m7_Path(ARTIFACT_DIR)
_m7_output.mkdir(parents=True, exist_ok=True)
_m7_pdf_path = _m7_output / "Module7_LUAD_Knowledge_Graph_Report.pdf"
_m7_ranked = biomarkers.sort_values(["combined_rank", "module2_rank"], kind="stable").copy()
_m7_top = _m7_ranked.head(5)
_m7_top_names = ", ".join(_m7_top["gene"].astype(str))
_m7_lead = _m7_ranked.iloc[0]
_m7_leading_ties = _m7_ranked.loc[
    (_m7_ranked["combined_priority"] - float(_m7_lead["combined_priority"])).abs() < 1e-12, "gene"
].astype(str).tolist()
_m7_leading_note = (
    f"{_m7_text(' and '.join(_m7_leading_ties))} share the leading priority "
    f"{_m7_number(_m7_lead['combined_priority'], 4)}; their display order reflects tie handling."
    if len(_m7_leading_ties) > 1 else
    f"{_m7_text(_m7_lead['gene'])} ranks first with combined priority "
    f"{_m7_number(_m7_lead['combined_priority'], 4)}."
)
_m7_selected = list(summary.get("selected_community", []))
if not _m7_selected:
    _m7_selected = sorted(next((c for c in comms if "COL11A1" in c), comms[0]))
_m7_selected = sorted(map(str, _m7_selected))
_m7_selected_genes = [name for name in _m7_selected if name in G and G.nodes[name].get("kind") == "gene"]
_m7_selected_pathways = [name for name in _m7_selected if name in G and G.nodes[name].get("kind") == "pathway"]

# The displayed path comes from the executed shared-phenotype query, then is
# checked against the exact local graph used for Neo4j loading and CSV export.
_m7_hit = next((row for row in repurposing_results
               if "topotecan" in str(row.get("repurposing_candidate", "")).lower()), None)
if _m7_hit is None:
    raise ValueError("The documented topotecan hypothesis requires its actual query result.")
_m7_symptoms = _m7_hit["shared_symptoms"]
if isinstance(_m7_symptoms, str):
    _m7_symptoms = [_m7_symptoms]
if not _m7_symptoms:
    raise ValueError("The actual query result has no shared phenotype.")
_m7_symptom = next((name for name in _m7_symptoms if "cough" in str(name).lower()), _m7_symptoms[0])
_m7_disease = globals().get("DISEASE")
if not _m7_disease:
    _m7_disease = next(name for name, data in G.nodes(data=True)
                       if data.get("kind") == "disease" and "adenocarcinoma" in name.lower())
_m7_path_nodes = [_m7_hit["repurposing_candidate"], _m7_hit["gene"],
                  _m7_hit["via_disease"], _m7_symptom, _m7_disease]
_m7_path_edges = [
    (_m7_path_nodes[0], "TARGETS", _m7_path_nodes[1]),
    (_m7_path_nodes[1], "ASSOCIATED_WITH", _m7_path_nodes[2]),
    (_m7_path_nodes[2], "HAS_PHENOTYPE", _m7_path_nodes[3]),
    (_m7_path_nodes[4], "HAS_PHENOTYPE", _m7_path_nodes[3]),
]
for _m7_src, _m7_rel, _m7_dst in _m7_path_edges:
    if not G.has_edge(_m7_src, _m7_dst) or G.edges[_m7_src, _m7_dst].get("rel") != _m7_rel:
        raise ValueError("A reported path edge is absent or differs from the actual graph.")
if not path_evidence:
    raise ValueError("Export supporting query evidence before generating the final report.")

_m7_directed_tied = bool(summary["directed_pagerank_tied"])
_m7_confidence_criteria = dict(summary.get("confidence_criteria", {}))
if not _m7_confidence_criteria:
    for _, _, _m7_edge_data in G.edges(data=True):
        if _m7_edge_data.get("confidence") and _m7_edge_data.get("confidence_basis"):
            _m7_confidence_criteria[_m7_edge_data["confidence"]] = _m7_edge_data["confidence_basis"]
_m7_confidence_note = " ".join(
    f"<b>{_m7_text(label.capitalize())}:</b> {_m7_text(basis)}"
    for label, basis in sorted(_m7_confidence_criteria.items())
)
_m7_pr_note = (
    "All 15 cohort genes have equal directed PageRank. Genes point toward disease and pathway nodes, "
    "while the incoming drug and variant edges enter contextual genes. The graph therefore provides "
    "no centrality discrimination among the original candidates. Their normalized centrality is set "
    "to the same neutral value, 0.5; this is a constant score component, not measured evidence."
    if _m7_directed_tied else
    "Directed PageRank differs among the original candidates. Its contribution reflects this graph's "
    "edge directions and selected annotation coverage, rather than biological causality."
)

_m7_node_items = sorted(summary["node_counts"].items())
_m7_edge_items = sorted(summary["relationship_counts"].items())
_m7_count_rows = [["<b>Node type</b>", "<b>n</b>", "<b>Relationship type</b>", "<b>n</b>"]]
for _m7_i in range(max(len(_m7_node_items), len(_m7_edge_items))):
    _m7_n = _m7_node_items[_m7_i] if _m7_i < len(_m7_node_items) else ("", "")
    _m7_e = _m7_edge_items[_m7_i] if _m7_i < len(_m7_edge_items) else ("", "")
    _m7_count_rows.append([_m7_text(_m7_n[0]), _m7_n[1], _m7_text(_m7_e[0]), _m7_e[1]])

_m7_page1 = [
    _m7_p("LUAD knowledge graph: evidence, targets and hypotheses", "title"),
    _m7_p("Ajay Kanaparthy | BIOT 6900 | Computational Lab 7 | November 2, 2026", "subtitle"),
    _m7_p("1. Schema and evidence", "heading"),
    _m7_p(
        f"The completed graph contains <b>{int(summary['node_count'])} nodes and "
        f"{int(summary['edge_count'])} relationships</b>. It combines the original 15 LUAD candidates "
        "with contextual EGFR and TOP1 genes. Disease, phenotype, pathway, drug and variant nodes "
        "connect molecular observations to interpretable clinical questions. Contextual genes do not "
        "enter the candidate ranking. Approved symbols and stable gene identifiers support consistent "
        f"merging {_m7_cite('hgnc')}."
    ),
    _m7_table(_m7_count_rows, [142, 35, 274, 35]),
    _m7_Spacer(1, 6),
    _m7_p(
        "Cohort gene-disease links represent measured RNA, protein and mutation associations, not "
        f"causal drivers {_m7_cite('module2')}. Reactome supports matrix organization, lectin "
        f"complement and EGFR pathway membership {_m7_cite('reactome_ecm', 'reactome_lectin', 'reactome_egfr')}. "
        "FDA labels support drug mechanisms and indication-specific context. NCI supports "
        "disease-symptom links; HPO standardizes cough and dyspnea "
        f"{_m7_cite('hpo_cough', 'hpo_dyspnea')}. ClinVar documents the external EGFR L858R variant, "
        f"without asserting its presence in this cohort {_m7_cite('clinvar_egfr')}."
    ),
    _m7_p(
        "Every exported relationship includes its source URL and confidence rationale. "
        f"{_m7_confidence_note} These categories concern the stored edge, not causal certainty. "
        "GWAS p-values are unavailable and are not inferred from RNA p-values."
    ),
]
_m7_focused_figure = (_m7_output / "luad_focused_paths.png").is_file()
_m7_figure_path = _m7_output / ("luad_focused_paths.png" if _m7_focused_figure else "my_knowledge_graph.png")
if not _m7_figure_path.is_file():
    raise FileNotFoundError("Create the actual graph figure before the final report.")
_m7_iw, _m7_ih = _m7_ImageReader(str(_m7_figure_path)).getSize()
_m7_scale = min(486 / _m7_iw, 178 / _m7_ih)
_m7_page1.extend([
    _m7_Image(str(_m7_figure_path), width=_m7_iw * _m7_scale, height=_m7_ih * _m7_scale),
    _m7_p(
        ("Figure 1. Focused drug, gene, disease and shared-phenotype paths from the actual graph, "
         "colored by node type. " if _m7_focused_figure else
         "Figure 1. The actual exported graph, colored by node type. ") +
        "The full PNG accompanies this report; nodes.csv and edges.csv retain every node and relationship.", "small"
    ),
])

_m7_rank_rows = [["<b>Rank</b>", "<b>Gene</b>", "<b>PageRank</b>", "<b>Omics</b>",
                   "<b>p_sm</b>", "<b>p_ab</b>", "<b>Combined</b>"]]
for _, _m7_row in _m7_top.iterrows():
    _m7_rank_rows.append([
        int(_m7_row["combined_rank"]), _m7_text(_m7_row["gene"]),
        _m7_number(_m7_row["pagerank"], 5), _m7_number(_m7_row["multiomics_score"], 4),
        _m7_number(_m7_row["p_sm"], 4), _m7_number(_m7_row["p_ab"], 4),
        _m7_number(_m7_row["combined_priority"], 4),
    ])
_m7_comparison = "; ".join(
    f"{row['gene']}: Module 2 rank {int(row['module2_rank'])}, Module 3 rank {int(row['module3_rank'])}"
    for _, row in _m7_top.head(3).iterrows()
)
_m7_znf_note = ""
_m7_znf_rows = _m7_ranked.loc[_m7_ranked["gene"] == "ZNF536"]
if not _m7_znf_rows.empty:
    _m7_z = _m7_znf_rows.iloc[0]
    _m7_znf_note = (
        f"ZNF536 illustrates the evidence limits: RNA log2 fold change {_m7_number(_m7_z['rna_lfc'], 2)} "
        f"and protein change {_m7_number(_m7_z['prot_lfc'], 2)} have opposite signs; "
        f"the protein estimate uses {int(_m7_z['prot_n_pairs'])} valid pairs. A large change in "
        "abundance does not establish a therapeutic dependency."
    )
_m7_ecm_names = {"COL6A6", "TNR", "TNXB", "COL11A1"}.intersection(_m7_selected)
if _m7_ecm_names and _m7_selected_pathways:
    _m7_mechanism = (
        "Its matrix-related genes and pathway suggest extracellular-matrix remodeling or altered "
        "stromal composition as a LUAD-associated mechanism. This interpretation concerns tissue "
        "biology; mixed cell populations could produce the same abundance pattern "
        + _m7_cite("reactome_ecm") + "."
    )
elif "MASP1" in _m7_selected and _m7_selected_pathways:
    _m7_mechanism = (
        "MASP1 and its pathway suggest a complement-related inflammatory mechanism; pathway "
        "membership alone does not demonstrate complement activation in LUAD "
        + _m7_cite("reactome_lectin") + "."
    )
elif "EGFR" in _m7_selected:
    _m7_mechanism = (
        "The EGFR group connects receptor signaling with an established targeted treatment. "
        "Its relevance depends on molecular context and cannot be generalized to all LUAD tumors "
        + _m7_cite("reactome_egfr", "fda_osimertinib") + "."
    )
else:
    _m7_mechanism = (
        "The grouping provides a candidate shared mechanism to inspect experimentally; the current "
        "annotations alone do not establish pathway activity."
    )
_m7_page2 = [
    _m7_p("2. Targets, evidence fusion and communities", "title"),
    _m7_p(_m7_pr_note),
    _m7_p(
        "Combined priority = 0.4C + 0.4M + 0.2T. M is the within-15 percentile of the original "
        "Module 2 multi-omics score; T is the within-15 percentile of max(p_sm, p_ab). C is normalized "
        "centrality, with 0.5 assigned when all values tie. The distinct raw p_sm and p_ab values "
        "remain visible below. These model-derived tractability priors are not calibrated probabilities "
        "of clinical success. Weights are declared teaching choices, rather than fitted or validated "
        f"clinical parameters {_m7_cite('module2', 'module3')}."
    ),
    _m7_table(_m7_rank_rows, [31, 65, 79, 78, 77, 77, 79]),
    _m7_Spacer(1, 6),
    _m7_p(
        f"The leading candidates are <b>{_m7_text(_m7_top_names)}</b>. "
        f"{_m7_leading_note} The retained omics evidence and relative tractability drive these positions. "
        "They should not be attributed to a differentiated PageRank signal when centrality ties. "
        f"Comparison: {_m7_text(_m7_comparison)}. Spearman rank correlations with Modules 2 and 3 "
        f"are {_m7_number(summary.get('rank_corr_module2'))} and "
        f"{_m7_number(summary.get('rank_corr_module3'))}, respectively."
    ),
    _m7_p(
        "ACTN2's leading rank reflects large decreases in both RNA and protein abundance, "
        "rather than evidence that inhibiting it would treat LUAD. COL6A6 connects to curated "
        "extracellular-matrix biology, supporting investigation of tissue composition and stromal "
        "remodeling. These cohort candidates require functional validation; EGFR provides separate "
        f"established, genotype-dependent treatment context {_m7_cite('module2', 'reactome_ecm', 'fda_osimertinib')}."
    ),
    _m7_p(_m7_znf_note),
    _m7_p("Detected community and interpretation", "heading"),
    _m7_p(
        f"Greedy modularity on the undirected graph found <b>{len(comms)} communities</b>. "
        f"The selected community contains {_m7_text('; '.join(_m7_selected))}. "
        f"Its genes are {_m7_text(', '.join(_m7_selected_genes) or 'none')}; its pathways are "
        f"{_m7_text('; '.join(_m7_selected_pathways) or 'none')}. {_m7_mechanism} "
        "Community membership depends on annotation choices, graph direction removal and sparse "
        "connectivity. It does not establish an experimentally validated pathway or patient subtype."
    ),
]

_m7_drug_sentences = "; ".join(
    f"{row['drug']} targets {row['gene']} ({row.get('status', 'status unavailable')})"
    for row in two_hop_results
)
_m7_path_rows = [["<b>Source node</b>", "<b>Relationship</b>", "<b>Destination node</b>"]]
for _m7_src, _m7_rel, _m7_dst in _m7_path_edges:
    _m7_path_rows.append([_m7_text(_m7_src), _m7_text(_m7_rel), _m7_text(_m7_dst)])
_m7_page3 = [
    _m7_p("3. Query results, hypothesis and limitations", "title"),
    _m7_p(
        f"The executed two-hop Drug → Gene → LUAD query returned: {_m7_text(_m7_drug_sentences)}. "
        "Approval refers to the source label's specified disease and molecular population, not every "
        f"LUAD case {_m7_cite('fda_osimertinib')}. The shared-phenotype query returned "
        f"{len(repurposing_results)} candidate row(s). Its actual topotecan route contains five nodes "
        "and four relationships; the last relationship is traversed in reverse."
    ),
    _m7_table(_m7_path_rows, [166, 154, 166]),
    _m7_Spacer(1, 6),
    _m7_p(
        "The topotecan label supports TOP1 inhibition and an SCLC treatment indication. TOP1's "
        "SCLC association therefore encodes therapeutic-target context, not an oncogenic driver "
        "claim. NCI documents cough and breathing difficulty in both clinical disease descriptions. "
        "A shared nonspecific symptom is weak evidence for drug transfer; this path does not show "
        f"that topotecan treats LUAD {_m7_cite('fda_topotecan', 'nci_sclc', 'nci_nsclc')}."
    ),
    _m7_p(
        "A testable hypothesis is that a molecularly defined LUAD subset could depend on TOP1 "
        "activity. Compare topotecan concentration-response curves in several LUAD models, "
        "nonmalignant lung controls and an SCLC comparator. Measure TOP1 engagement, DNA-damage "
        "markers and cell survival. Use an independent TOP1 perturbation and an appropriate rescue "
        "strategy to assess on-target dependence. Sensitivity, selectivity and reproducibility would "
        "be required before animal studies or any clinical interpretation; cough alone should not "
        "select treatment."
    ),
    _m7_p("Uncertainty and checks", "heading"),
    _m7_p(
        "Cohort abundance associations may reflect tumor purity, stromal content or confounding. "
        "Selected pathway annotations are incomplete, and centrality changes with graph design. "
        "Within-list percentiles can amplify small original score differences. Tractability estimates "
        "come from a teaching classifier; COL6A6 and TNXB have truncated embedding representations "
        f"{_m7_cite('module3')}. Module 4 pocket results for SCN7A and MASP1 add structural context "
        f"without proving ligand binding or LUAD dependence {_m7_cite('module4')}. "
        "The optional patient layer is empty because Module 6 ages and disease labels are simulated. "
        "Module 5 CYP2C19 pharmacogenomic variants are not relabeled as LUAD driver mutations."
    ),
    _m7_p(
        "AI assisted identifier harmonization, source review, notebook coding and report drafting. "
        "Checks compared the original CSVs, official gene identifiers, source-specific biological "
        "claims, executed Neo4j query results and every reported path edge with the exported graph. "
        "Further progress requires independent LUAD cohorts, cell-type-resolved measurements, "
        "binding assays and functional perturbation. These checks support reproducibility, while "
        "the graph remains a hypothesis-generating synthesis."
    ),
]

_m7_page4 = [
    _m7_p("References", "title"),
    _m7_p("Web resources were checked on November 2, 2026. FDA citations identify the specific "
          "label versions used; they are not described as the latest versions.", "small"),
]
for _m7_key in sorted(_m7_required_sources,
                      key=lambda key: (_m7_author(SOURCES[key]).lower(),
                                       str(SOURCES[key].get('year', 'n.d.')),
                                       str(SOURCES[key]['title']).lower())):
    _m7_src = SOURCES[_m7_key]
    _m7_url = str(_m7_src.get("url", ""))
    _m7_link = (f' <link href={_m7_quoteattr(_m7_url)} color="black">{_m7_text(_m7_url)}</link>'
                if _m7_url.startswith(("http://", "https://")) else
                (" " + _m7_text(_m7_url) if _m7_url else ""))
    _m7_page4.append(_m7_p(
        f"{_m7_text(_m7_author(_m7_src))}. ({_m7_text(_m7_citation_year[_m7_key])}). "
        f"<i>{_m7_text(_m7_src['title'])}</i>.{_m7_link}", "reference"
    ))

def _m7_footer(canvas, document):
    canvas.saveState()
    canvas.setFillColor(_m7_colors.black)
    canvas.setFont("Helvetica", 8)
    canvas.drawString(document.leftMargin, 23, "BIOT 6900 · Module 7 · LUAD")
    canvas.drawRightString(_m7_letter[0] - document.rightMargin, 23, f"{document.page} / 4")
    canvas.restoreState()

_m7_doc = _m7_Doc(str(_m7_pdf_path), pagesize=_m7_letter,
                   leftMargin=44, rightMargin=44, topMargin=35, bottomMargin=38,
                   title="Module 7 LUAD Knowledge Graph Report", author="Ajay Kanaparthy")
_m7_story = []
for _m7_page_index, _m7_content in enumerate((_m7_page1, _m7_page2, _m7_page3, _m7_page4)):
    if _m7_page_index:
        _m7_story.append(_m7_PageBreak())
    # Fail rather than silently shrink text or produce an accidental fifth page.
    _m7_story.append(_m7_KeepInFrame(_m7_doc.width - 12, _m7_doc.height - 12,
                                    _m7_content, mode="error", hAlign="LEFT", vAlign="TOP"))
_m7_doc.build(_m7_story, onFirstPage=_m7_footer, onLaterPages=_m7_footer)

try:
    import pymupdf as _m7_fitz
except ImportError:
    _m7_pdf_pages = 4  # Explicit single-page frames and three PageBreaks above.
else:
    with _m7_fitz.open(str(_m7_pdf_path)) as _m7_document:
        _m7_pdf_pages = len(_m7_document)
    if _m7_pdf_pages != 4:
        raise ValueError(f"The final report has {_m7_pdf_pages} pages; exactly four are required.")

_m7_deliverables = [
    "BIOT6900_Module7_Starter.ipynb", "my_knowledge_graph.png", "luad_focused_paths.png",
    "nodes.csv", "edges.csv", "biomarkers.csv", "communities.csv", "query_evidence.csv",
    "analysis_summary.json", "sources.csv", "Module7_LUAD_Knowledge_Graph_Report.pdf", "README.md",
]
_m7_file_lines = "\n".join(f"- `{name}`" for name in _m7_deliverables
                             if (_m7_output / name).exists() or name == "README.md")
_m7_node_md = "\n".join(f"| {name} | {count} |" for name, count in _m7_node_items)
_m7_edge_md = "\n".join(f"| {name} | {count} |" for name, count in _m7_edge_items)
_m7_path_md = "\n".join(f"- `{src}` —`{rel}`→ `{dst}`" for src, rel, dst in _m7_path_edges)
_m7_source_md = "\n".join(
    f"- {SOURCES[key]['title']}: {SOURCES[key].get('url', '')}" for key in _m7_required_sources
)
_m7_readme = f"""# BIOT 6900 Module 7 — LUAD

Ajay Kanaparthy. The professor's original `BIOT6900_Module7_Starter.ipynb` contains the completed analysis; the reference notebooks are not submission replacements.

## Outputs

{_m7_file_lines}

The report has {_m7_pdf_pages} pages including references. It uses the executed graph, query results, candidate rankings and community output.

## Graph and actual results

The assignment graph contains **{int(summary['node_count'])} nodes and {int(summary['edge_count'])} relationships**. Counts describe this analysis rather than unrelated data already in AuraDB.

| Node type | Count |
|---|---:|
{_m7_node_md}

| Relationship type | Count |
|---|---:|
{_m7_edge_md}

The original 15 candidates are ranked separately from contextual EGFR and TOP1 genes. The top five combined priorities are **{_m7_top_names}**. Directed PageRank tied across all 15 cohort candidates: **{_m7_directed_tied}**. Priority is `0.4 * centrality + 0.4 * within-list omics percentile + 0.2 * within-list max(p_sm, p_ab) percentile`; a tied centrality component is set equally to 0.5. Both tractability priors remain separate in `biomarkers.csv`. Weights are descriptive teaching choices, not clinically optimized parameters.

The two-hop query returned: {_m7_drug_sentences}.

One actual shared-phenotype path has five nodes and four relationships (the final edge is traversed in reverse):

{_m7_path_md}

This topotecan path suggests a testable hypothesis; shared symptoms do not establish LUAD treatment efficacy. TOP1-to-SCLC represents therapeutic-target context. Greedy modularity found **{len(comms)} communities**. The interpreted community contains: {'; '.join(_m7_selected)}. Full membership appears in `communities.csv`.

## Run the original notebook

1. Select the Python `biot6900` kernel with the required packages installed.
2. Store Aura credentials locally in `Module_7/.env`, using `NEO4J_URI`, `NEO4J_USERNAME` and `NEO4J_PASSWORD`; quoted values are supported. This file is git-ignored. Environment variables override local configuration. A hidden password prompt is used only if the URI and username are available and the password is missing. Never publish credentials or paste them into outputs.
3. Use the Aura home database by default. Set optional `NEO4J_DATABASE` only when its actual name is known; do not assume the database is named `neo4j`.
4. Run **Kernel → Restart & Run All**. The connection cell verifies connectivity before database work; missing or invalid settings stop execution.
5. Inspect the query tables, figure, CSVs and final four-page PDF. Output paths resolve to `Module_7` when launched from the repository or module directory.

The notebook uses `MERGE` and uniqueness constraints. It does not reset or delete the existing database.

## Existing data and limitations

- Module 2: `module_2/targets_luad.csv` and `module_2/data/luad_integrated_gene_level.csv` provide measured molecular associations and harmonized identifiers. GWAS p-values are unavailable and remain missing.
- Module 3: `module_3/targets_luad_w3.csv` provides separate model-derived `p_sm` and `p_ab` tractability priors and the previous ranking. These are not calibrated probabilities of LUAD treatment success; COL6A6 and TNXB have truncated embeddings.
- Module 4: `module_4/results/verified_structure_summary.csv` provides SCN7A/MASP1 structural context. Predicted pockets do not prove binding or functional LUAD dependence.
- Module 5: CYP2C19 variants concern pharmacogenomics and are not labeled as LUAD driver mutations. The graph's EGFR L858R is independently sourced external context, not an asserted cohort variant.
- Module 6: Genotypes are real, while ages and disease labels are simulated. The optional patient layer is empty.
- Curated pathway membership and observational abundance associations do not establish causality. Shared cough/dyspnea are nonspecific; PageRank and communities depend on graph annotation and edge directions. Qualitative confidence documents support for individual edges rather than clinical certainty.
- AI assisted source review, identifier harmonization, coding and report drafting. Results were checked against original data, official sources, executed Neo4j queries and exported relationships.

## Sources

Every relationship records `source_url`, confidence and its rationale in `edges.csv`; the source catalog appears in `sources.csv`. Confidence criteria: {'; '.join(f'{label}: {basis}' for label, basis in sorted(_m7_confidence_criteria.items()))}. These categories describe edge support rather than causal certainty. The report provides APA-style citations and references.

{_m7_source_md}
"""
(_m7_output / "README.md").write_text(_m7_readme, encoding="utf-8")
print(f"Saved {_m7_pdf_path.name} ({_m7_pdf_pages} pages) and README.md.")


Saved Module7_LUAD_Knowledge_Graph_Report.pdf (4 pages) and README.md.
